# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Note: You can access .metadata as an object with .name and .description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs.

**Note:** All Croissant entities (record sets, fields, columns) are referenced by their `@id` fields.

In [ ]:
# Explore record sets and their fields
metadata_json = dataset.metadata.to_json()

record_sets = metadata_json.get('recordSet', [])
if not record_sets:
    print("No record sets defined in the schema.")
else:
    for rec in record_sets:
        rec_id = rec.get('@id', 'N/A')
        rec_name = rec.get('name', 'N/A')
        print(f"RecordSet ID: {rec_id}; Name: {rec_name}")
        fields = rec.get('field', [])
        # fields can be a dict if only one field, or list
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            # field could be a reference or an embedded object
            if isinstance(field, dict):
                f_id = field.get('@id', 'N/A')
                f_name = field.get('name', field.get('label', 'N/A'))
                print(f"  - Field ID: {f_id}; Name: {f_name}")
            else:
                print(f"  - Field ID: {field}")
else:
    # If no recordSet, check if distribution contains tabular files with fields
    distributions = metadata_json.get('distribution', [])
    print("Trying to infer structure from distribution entries...")
    for dist in distributions:
        dist_id = dist.get('@id', 'N/A') if isinstance(dist, dict) else dist
        print(f"Distribution ID: {dist_id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

**Note:** For this schema, we'll extract all top-level record sets available. If none are defined, we attempt to extract tabular data from the available distributions.

In [ ]:
# Find available record set IDs
metadata_json = dataset.metadata.to_json()
record_sets = metadata_json.get('recordSet', [])

# If the schema does not provide recordSet entries, we try to infer from 'distribution'.
record_set_ids = []
if record_sets:
    # Each recordSet may be an object or a reference
    for rec in record_sets:
        if isinstance(rec, dict):
            record_set_ids.append(rec.get('@id'))
        else:
            record_set_ids.append(rec)

else:
    # If no record sets, infer them from distribution
    distributions = metadata_json.get('distribution', [])
    if distributions:
        for dist in distributions:
            if isinstance(dist, dict):
                # Use the distribution @id as record set id, as fallback
                record_set_ids.append(dist.get('@id'))
            else:
                record_set_ids.append(dist)
    else:
        print("No record sets or distribution entries found.")

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for record set {record_set_id} with {len(df)} records and columns: {df.columns.tolist()}")
        else:
            print(f"No records found for record set {record_set_id}.")
    except Exception as e:
        print(f"Error loading records for {record_set_id}: {e}")

# Show example DataFrame (from first loaded record set)
if dataframes:
    first_rs = next(iter(dataframes))
    print(f"First record set loaded: {first_rs}")
    print("Columns:", dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())
else:
    print("No dataframes were loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

For this dataset, let's:
- Filter proteins with a high peptide count
- Normalize the molecular weight field
- Group by description or accession, if available

All field references use their original `@id` or column names obtained above.

In [ ]:
import numpy as np
# Select a record set to analyze

if not dataframes:
    print("No data loaded for analysis.")
else:
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id]
    # Inspect columns for possible numeric fields
    print("Columns:", df.columns.tolist())
    # Try to pick likely numeric fields (e.g., 'peptide_count', 'MW', 'coverage_pct')
    candidate_numeric_fields = [col for col in df.columns if any(key in col.lower() for key in ['count', 'mw', 'weight', 'coverage', 'abundance', 'intensity'])]
    print('Candidate numeric fields:', candidate_numeric_fields)
    
    if candidate_numeric_fields:
        numeric_field = candidate_numeric_fields[0]
        print(f"Selected numeric field for filtering: {numeric_field}")
        # Remove missing values, convert to numeric
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = np.nanquantile(df[numeric_field], 0.75)  # top 25% for demonstration
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
        display(filtered_df.head())

        # Normalize selected numeric field
        field_norm = f"{numeric_field}_normalized"
        filtered_df[field_norm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, field_norm]].head())

        # Try grouping by a categorical field
        candidate_group_fields = [col for col in df.columns if any(key in col.lower() for key in ['description', 'accession', 'protein']) and col != numeric_field]
        group_field = candidate_group_fields[0] if candidate_group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
            print(f"Grouped data by {group_field} (showing mean {numeric_field}):")
            display(grouped_df.head())
        else:
            print("No categorical field selected for grouping.")
    else:
        print("No numeric fields found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id]
    # Try to use the normalized or numeric fields from before
    candidate_numeric_fields = [col for col in df.columns if any(key in col.lower() for key in ['count', 'mw', 'weight', 'coverage', 'abundance', 'intensity'])]
    if candidate_numeric_fields:
        f = candidate_numeric_fields[0]
        df[f] = pd.to_numeric(df[f], errors='coerce')
        plt.figure(figsize=(8,4))
        sns.histplot(df[f].dropna(), kde=True)
        plt.title(f'Distribution of {f}')
        plt.xlabel(f)
        plt.show()

        if len(candidate_numeric_fields) > 1:
            f2 = candidate_numeric_fields[1]
            df[f2] = pd.to_numeric(df[f2], errors='coerce')
            plt.figure(figsize=(5,5))
            sns.scatterplot(x=df[f], y=df[f2])
            plt.title(f'Scatter plot: {f} vs {f2}')
            plt.xlabel(f)
            plt.ylabel(f2)
            plt.show()
    else:
        print("No suitable numeric fields for visualization.")
else:
    print("No data loaded for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded the Mass Spectrometry EV dataset using `mlcroissant`.
- We inspected available record sets, explored field structures, and extracted tabular data.
- Basic filtering and normalization on numeric molecular and abundance properties allowed for preliminary analysis.
- Grouping by protein accession or description can facilitate biomarker discovery and protein pattern studies.
- The field and record set IDs are always referenced via their canonical `@id` values, following Croissant standards.

For deeper proteomics or machine learning analysis, further data integration and cleansing steps may be applied.